### Model training 
In the previous notebook we performed hyperparamer tuning. Now we are ready to train the embeddings model based on the best hyper parameters and export to model repository.
![Training Dataset](./images/experiment_td.png)

Here as well we are going to use StellarGraph library to compute node embeddings. StellarGraph supports loading data via Pandas DataFrames, NumPy arrays, Neo4j and NetworkX graphs. 

---
**NOTE**:

Loading large scale dataset in to StellarGraph for training can not be handled with above mentioned fameworks. It will require loading data using frameworks such as `tf.data`. 

If your training datasets measure from couple of GB to 100s of GBs or even TBs contact us at Logical Clocks and we will help you to setup distributed training pipelines. 

---

### Define hopsworks experiments wrapper function and put all the training logic there. 

In [ ]:
# Imports and setup for local execution
import os
import json
import uuid
import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")
MODELS_PATH = os.path.join(BASE_PATH, "models")

# Override paths when running via pipeline (artifacts_dir injected by papermill)
try:
    if artifacts_dir:
        TRAINING_DATA_PATH = os.path.join(artifacts_dir, "data")
        MODELS_PATH = os.path.join(artifacts_dir, "models")
except NameError:
    pass

os.makedirs(MODELS_PATH, exist_ok=True)

print(f"Training data path: {TRAINING_DATA_PATH}")
print(f"Models will be saved to: {MODELS_PATH}")

## Use above experiments wrapper function to conduct hops training experiments.

In [2]:
# Load best hyperparameters from previous notebook
best_hyperparams_path = os.path.join(RESOURCES_PATH, "embeddings_best_hp.json")

with open(best_hyperparams_path, 'r') as f:
    best_hyperparams = json.load(f)

print(f"Loaded best hyperparameters: {best_hyperparams}")

Loaded best hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}


In [3]:
# Load training data
print("Loading training data...")
node_pdf = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "node_td.csv"))
edge_pdf = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "edges_td.csv"))
alert_nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv"))

print(f"Nodes: {len(node_pdf)}, Edges: {len(edge_pdf)}, Alert nodes: {len(alert_nodes_df)}")

# Build NetworkX graph
print("\nBuilding NetworkX directed graph...")
G = nx.from_pandas_edgelist(
    edge_pdf, 
    source='source', 
    target='target', 
    edge_attr=['tx_type', 'base_amt'],
    create_using=nx.DiGraph()
)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Loading training data...
Nodes: 7347, Edges: 438386, Alert nodes: 7347

Building NetworkX directed graph...
Graph: 7347 nodes, 17070 edges


In [4]:
# Train Node2Vec model with best hyperparameters
walk_number = best_hyperparams['walk_number']
walk_length = best_hyperparams['walk_length']
emb_size = best_hyperparams['emb_size']

print(f"Training Node2Vec with: walk_number={walk_number}, walk_length={walk_length}, emb_size={emb_size}")

# Create and train Node2Vec model
node2vec = Node2Vec(
    G, 
    dimensions=emb_size, 
    walk_length=walk_length, 
    num_walks=walk_number,
    p=0.5,
    q=2.0,
    workers=4,
    quiet=False
)

print("\nTraining Word2Vec on random walks...")
model = node2vec.fit(window=10, min_count=1, batch_words=4)
print("Training complete!")

Training Node2Vec with: walk_number=2, walk_length=2, emb_size=32


Generating walks (CPU: 2): 100%|██████████| 1/1 [00:00<00:00, 76.28it/s]
Generating walks (CPU: 3): 0it [00:00, ?it/s]
Generating walks (CPU: 4): 0it [00:00, ?it/s]



Training Word2Vec on random walks...
Training complete!


### Managing experiments
Experiments service provides a unified view of all the experiments run using the `experiment` module.
<br>
As demonstrated in the gif it provides general information about the experiment and the resulting metric. Experiments can be visualized meanwhile or after training in a TensorBoard.
<br>
<br>
![Image7-Monitor.png](./images/experiments.png)

In [5]:
# Extract embeddings for all nodes
print("Extracting node embeddings...")

embeddings_dict = {}
for node in G.nodes():
    node_str = str(node)
    if node_str in model.wv:
        embeddings_dict[node_str] = model.wv[node_str]

print(f"Generated embeddings for {len(embeddings_dict)} nodes")

# Create embeddings dataframe
embeddings_df = pd.DataFrame.from_dict(embeddings_dict, orient='index')
embeddings_df.index.name = 'node_id'
embeddings_df.columns = [f'emb_{i}' for i in range(emb_size)]
embeddings_df = embeddings_df.reset_index()

print(f"\nEmbeddings shape: {embeddings_df.shape}")
embeddings_df.head()

Extracting node embeddings...
Generated embeddings for 7347 nodes

Embeddings shape: (7347, 33)


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31
0,3aa9646b,0.016530,-0.010947,-0.000496,-0.000766,0.028229,-0.007027,0.013945,-0.009410,0.007632,...,-0.012160,-0.022671,-0.005034,-0.024899,-0.022038,-0.021844,-0.009677,0.020635,0.008849,-0.024740
1,1e46e726,0.006591,0.013344,-0.015142,0.026666,0.026163,0.028550,-0.030208,-0.028693,0.022439,...,-0.004233,-0.011421,0.019160,0.018987,0.021968,-0.011300,0.030992,0.027277,-0.012025,-0.010329
2,49203bc3,0.005050,0.002158,-0.025958,0.010061,-0.030157,0.024989,-0.020235,0.019330,-0.028948,...,0.009855,-0.000529,0.009421,-0.022867,0.015096,0.002769,0.014327,-0.019856,-0.015230,-0.025997
3,a74d1101,0.020242,0.017416,0.028806,0.027612,-0.000559,0.013644,-0.029239,0.025620,0.031111,...,-0.016263,0.002794,0.010684,-0.031003,0.005417,0.019303,-0.019357,0.029952,-0.028007,-0.023648
4,616d4505,0.029466,0.012671,-0.006395,0.021655,0.031388,-0.016556,0.018658,-0.017666,0.013849,...,-0.011222,-0.022778,-0.022365,0.010624,-0.014411,0.018756,0.009777,-0.020094,0.006874,-0.022809


In [6]:
# Evaluate embeddings quality using is_sar classification
print("Evaluating embeddings quality...")

# Get embeddings for nodes in alert_nodes_df
X = []
y = []
for _, row in alert_nodes_df.iterrows():
    node_id = str(row['id'])
    if node_id in embeddings_dict:
        X.append(embeddings_dict[node_id])
        y.append(row['is_sar'])

X = np.array(X)
y = np.array(y)

print(f"Evaluation dataset: {len(X)} nodes")

# Train/test split and evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nEmbedding Evaluation Accuracy: {accuracy:.4f}")
metrics = {'accuracy': accuracy}

Evaluating embeddings quality...
Evaluation dataset: 7347 nodes

Embedding Evaluation Accuracy: 0.8891


In [7]:
# Save model and embeddings locally (replaces Hopsworks model registry)
model_id = str(uuid.uuid4())[:8]
model_dir = os.path.join(MODELS_PATH, f"node_embeddings_{model_id}")
os.makedirs(model_dir, exist_ok=True)

# Save Word2Vec model
model_path = os.path.join(model_dir, "node2vec_model.model")
model.save(model_path)
print(f"Saved Node2Vec model to: {model_path}")

# Save classifier
clf_path = os.path.join(model_dir, "classifier.joblib")
joblib.dump(clf, clf_path)
print(f"Saved classifier to: {clf_path}")

# Save embeddings as CSV
embeddings_path = os.path.join(model_dir, "node_embeddings.csv")
embeddings_df.to_csv(embeddings_path, index=False)
print(f"Saved embeddings to: {embeddings_path}")

# Save metrics and hyperparameters
metadata = {
    'hyperparameters': best_hyperparams,
    'metrics': metrics,
    'num_nodes': len(embeddings_dict),
    'embedding_dim': emb_size
}
metadata_path = os.path.join(model_dir, "metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata to: {metadata_path}")

# Also save embeddings to training_data for next notebooks
embeddings_df.to_csv(os.path.join(TRAINING_DATA_PATH, "node_embeddings.csv"), index=False)
print(f"\nAlso saved embeddings to: {TRAINING_DATA_PATH}/node_embeddings.csv")

print(f"\n{'='*50}")
print(f"Model saved to: {model_dir}")
print(f"Accuracy: {accuracy:.4f}")
print(f"{'='*50}")

Saved Node2Vec model to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_0cf6becf/node2vec_model.model
Saved classifier to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_0cf6becf/classifier.joblib
Saved embeddings to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_0cf6becf/node_embeddings.csv
Saved metadata to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_0cf6becf/metadata.json

Also saved embeddings to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/node_embeddings.csv

Model saved to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_0cf6becf
Accuracy: 0.8891
